# AlephLLM — Mini-Beatrix mission control

Skeleton entrypoint: the repo carries all functionality; this notebook only triggers it.
Each session **resumes where the last one stopped** (manifest + resume state pulled from the HF training repo).
Stop training any time with the stop button — the interrupt handler checkpoints and uploads before exiting.

**Updating after a repo patch:** rerun cell 1, and if it prints `RESTART REQUIRED`, do `Runtime > Restart session`, then rerun from cell 1. A live Python runtime cannot hot-swap an already-imported package.

In [ ]:
# 1 — install / update the repo
%pip install -q --upgrade git+https://github.com/AbstractEyes/alephllm
import importlib.metadata, sys
installed = importlib.metadata.version("geolip-alephllm")
if "geolip.alephllm" in sys.modules:
    import geolip.alephllm as al
    if al.__version__ != installed:
        print(f"RESTART REQUIRED: runtime has {al.__version__}, installed {installed} — Runtime > Restart session")
    else:
        print("geolip.alephllm", al.__version__, "(already loaded)")
else:
    import geolip.alephllm as al
    print("geolip.alephllm", al.__version__)

In [ ]:
# 2 — prep: token, craft, resume point
from google.colab import userdata
from geolip.alephllm import prepare

PRESET = "mini-beatrix-1"   # see geolip.alephllm.PRESETS for the ladder
run = prepare(PRESET, hf_token=userdata.get("HF_TOKEN"))

In [ ]:
# 3 — train (interrupt-safe; checkpoints + manifest + tensorboard upload as it goes)
run.train(max_hours=11.5)

In [ ]:
# 4 — eval: val bpb, toggle ledger, canaries, full structural census
run.evaluate()

In [ ]:
# 5 (optional) — sample from the current weights
import torch
prompt = "The "
ids = torch.tensor([list(prompt.encode())], device=run.device)
out = run.model.generate(ids, max_new=200)
print(run.tokenizer.decode(out[0].tolist()))
run.model.train()

In [ ]:
# 6 (optional) — full smoke/test array (fast; run after repo patches)
!python -m geolip.alephllm.tests.smoke

In [ ]:
# 6b — ANNEAL PHASE (run after fineweb_extended completes, BEFORE the
# second chat arm): ~2B tokens on the shifted mix — fineweb 45% /
# cosmopedia 20% / tinystories 15% / SODA dialogue 12% / beatrix
# template texture 5% / recall garnish 3%. ~5.3h at current throughput.
run.manifest.phases.append(dict(name="anneal_mix", dataset="anneal-mix",
                                planned_tokens=2_000_000_000,
                                tokens_done=0, status="planned"))
run.manifest.note("anneal phase appended: two-arm experiment — arm A on the"
                  " pre-anneal locked core, arm B on the post-anneal core")
run.train(max_hours=11.5)


In [ ]:
# 7 — FIRST CHAT CONDITIONING (run after the pretrain locks)
# Trains the chat arm on the frozen core (smoltalk everyday + persona),
# gauges core preservation + toggle law, ships the arm to the training repo.
%pip install -q git+https://github.com/AbstractEyes/amoe-lora
from geolip.alephllm.chat_sft import run_chat_conditioning
report = run_chat_conditioning(hf_token=userdata.get("HF_TOKEN"),
                               steps=1200, mode="arm")  # mode="both" adds the full-SFT control
